In [ ]:
!pip install tqdm --quiet

In [ ]:
from google.colab import files

print("Please upload your netflix_titles.csv file:")
uploaded = files.upload()

print("File Uploaded Successfully!!")

Please upload your netflix_titles.csv file:


Saving netflix_titles.csv to netflix_titles.csv
File Uploaded Successfully!!


In [ ]:
import pandas as pd
import requests
import time
from tqdm import tqdm

TMDB_API_KEY = "3e84ffbf60cdbf3bd2823190fedeb3fc"

print("Libraries loaded and API key set!!")

Libraries loaded and API key set!!


In [ ]:
print("Loading Netflix Data...")

df = pd.read_csv("netflix_titles.csv")

print(f"Loaded {len(df)} rows and {len(df.columns)} columns")
print("\First 5 rows:")
df.head()

Loading Netflix Data...
Loaded 8807 rows and 12 columns
\First 5 rows:


<>:6: SyntaxWarning: invalid escape sequence '\F'
<>:6: SyntaxWarning: invalid escape sequence '\F'
/tmp/ipykernel_2704/3480586863.py:6: SyntaxWarning: invalid escape sequence '\F'
  print("\First 5 rows:")


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
print("Cleaning Data...")

df = df.drop_duplicates(subset="show_id")

df["director"] = df["director"].fillna("Unknown")
df["cast"] = df["cast"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
df["rating"] = df["rating"].fillna("Not Rated")

df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), errors="coerce")

df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month

def get_duration_number(duration):
  if pd.isna(duration):
    return None
  return int(str(duration).split(" ")[0])

def get_duration_unit(duration):
  if pd.isna(duration):
    return None
  if "min" in str(duration):
    return "mintutes"
  if "Season" in str(duration):
    return "seasons"
  return None

df["duration_value"] = df["duration"].apply(get_duration_number)
df["duration_unit"] = df["duration"].apply(get_duration_unit)

df["description"] = df["description"].fillna("").str.strip()

print("Cleaning done!")
print(f"\nMissing values remaining:")
print(df.isnull().sum())

Cleaning Data...
Cleaning done!

Missing values remaining:
show_id            0
type               0
title              0
director           0
cast               0
country            0
date_added        10
release_year       0
rating             0
duration           3
listed_in          0
description        0
year_added        10
month_added       10
duration_value     3
duration_unit      3
dtype: int64


In [ ]:
from numpy import empty
def get_tmdb_info(title, release_year, content_type):
  endpoint = "movie" if content_type == "Movie" else "tv"

  url = f"https://api.themoviedb.org/3/search/{endpoint}"

  params = {
      "api_key": TMDB_API_KEY,
      "query": title,
      "year": release_year,
  }

  try:
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    results = data.get("results", [])

    if results:
      top = results[0]
      return {
          "tmdb_id": top.get("id"),
          "tmdb_popularity": top.get("popularity"),
          "tmdb_vote_avg": top.get("vote_average"),
          "tmdb_vote_count": top.get("vote_count")
      }

  except Exception as e:
    print(f"Skipped: {title} {e}")

  return {
      "tmdb_id": None,
      "tmdb_popularity": None,
      "tmdb_vote_avg": None,
      "tmdb_vote_count": None,
  }

LIMIT = 100

print(f"Fetching TMDB info for {LIMIT} titles...")
print("(A progress br will appear below)")

tmdb_results = []

for i, row in tqdm(df.head(LIMIT).iterrows(), total = LIMIT):
  info = get_tmdb_info(
      title = row["title"],
      release_year = row["release_year"],
      content_type = row["type"],
  )
  tmdb_results.append(info)
  time.sleep(0.25)

empty = {"tmdb_id":None, "tmdb_popularity":None, "tmdb_vote_avg":None, "tmdb_vote_count":None}
tmdb_results += [empty] * (len(df) - LIMIT)

tmdb_df = pd.DataFrame(tmdb_results, index=df.index)
df = pd.concat([df, tmdb_df], axis = 1)

matched = df["tmdb_id"].notna().sum()
print(f"Done! Got TMDB data for {matched} out of {LIMIT} titles")

Fetching TMDB info for 100 titles...
(A progress br will appear below)


100%|██████████| 100/100 [00:39<00:00,  2.55it/s]

Done! Got TMDB data for 95 out of 100 titles


In [ ]:
print("Creating new Columns...")

df["release_date"] = pd.to_datetime(df["release_year"].astype(str) + "-01-01", errors = "coerce")

df["days_to_netflix"] = (df["date_added"] - df["release_date"]).dt.days


df["is_likely_original"] = df["days_to_netflix"].between(0, 365)

df["cast_size"] = df["cast"].apply(
    lambda x: 0 if x == "Unknown" else len(str(x).split(","))
)

df["genre_count"] = df["listed_in"].apply(
    lambda x: len(str(x).split(","))
)

df["primary_genre"] = df["listed_in"].apply(
    lambda x: str(x).split(",")[0].strip()
)

df["primary_country"] = df["country"].apply(
    lambda x: str(x).split(",")[0].strip()
)

df["decade"] = (df["release_year"] // 10 * 10).astype("Int64").astype(str) + "s"

print("New columns created!")
print("Preview of final table:")
df[["title", "type", "primary_genre", "primary_country", "year_added", "tmdb_popularity", "tmdb_vote_avg"]].head(10)

Creating new Columns...
New columns created!
Preview of final table:


,title,type,primary_genre,primary_country,year_added,tmdb_popularity,tmdb_vote_avg
0,Dick Johnson Is Dead,Movie,Documentaries,United States,2021.0,0.8701,7.100
1,Blood & Water,TV Show,International TV Shows,South Africa,2021.0,6.4325,7.800
2,Ganglands,TV Show,Crime TV Shows,Unknown,2021.0,3.4822,7.138
3,Jailbirds New Orleans,TV Show,Docuseries,Unknown,2021.0,0.0791,5.600
4,Kota Factory,TV Show,International TV Shows,India,2021.0,5.8611,8.022
5,Midnight Mass,TV Show,TV Dramas,Unknown,2021.0,8.1101,7.535
6,My Little Pony: A New Generation,Movie,Children & Family Movies,Unknown,2021.0,2.6947,7.700
7,Sankofa,Movie,Dramas,United States,2021.0,0.7914,5.800
8,The Great British Baking Show,TV Show,British TV Shows,United Kingdom,2021.0,1.7933,6.900
9,The Starling,Movie,Comedies,United States,2021.0,1.5355,6.915


In [ ]:
df.to_csv("netflix_cleaned.csv", index = False)
print(f"Saved netflix_cleaned.csv ({len(df)} rows, {len(df.columns)} columns)")

genre_rows = []
for _, row in df.iterrows():
  for genre in str(row["listed_in"]).split(","):
    genre_rows.append({
        "show_id": row["show_id"],
        "title": row["title"],
        "type": row["type"],
        "genre": genre.strip(),
    })

genre_df = pd.DataFrame(genre_rows)
genre_df.to_csv("netflix_genres.csv", index = False)
print(f"Saved netflix_genres.csv ({len(genre_df)} rows)")

print("\nDownoading files to your computer...")
files.download("netflix_cleaned.csv")
files.download("netflix_genres.csv")

print("\n ETL Pipeline Complete!!")
print("You now have two files ready for EDA and Tableau")

Saved netflix_cleaned.csv (8807 rows, 28 columns)
Saved netflix_genres.csv (19323 rows)

Downoading files to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 ETL Pipeline Complete!!
You now have two files ready for EDA and Tableau
